In [0]:

#metadata_setup notebook code

#dynamic catalog name usage
dbutils.widgets.text("catalog_name", "food_inspection")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
# Step 1: Create all schemas
schemas = ["raw_zone","bronze", "silver", "gold", "metadata"]

for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema}")
    print(f"{schema} created")

# Set current catalog
spark.sql(f"USE CATALOG {catalog_name}")
print(f"Using catalog: {catalog_name}")

In [0]:
%sql
-- Step 2: Create tables (safe for new users)

-- pipeline_control: master metadata table
-- One row per source dataset

CREATE TABLE IF NOT EXISTS metadata.pipeline_control (
    table_name      STRING,
    file_name       STRING,
    active_flag     STRING,
    city            STRING,
    created_date    DATE,
    modified_date   DATE
);

-- execution_log: audit trail for Raw zone writes
-- One row appended per dataset per pipeline run

CREATE TABLE IF NOT EXISTS metadata.execution_log (
    table_name          STRING,
    city                STRING,
    execution_time      TIMESTAMP,
    status              STRING,
    source_row_count    BIGINT,
    target_row_count    BIGINT,
    file_location       STRING,
    created_date        DATE
);

-- dqx_execution_log: DQX validation metrics per Silver run
-- One row appended per dataset per pipeline run
CREATE TABLE IF NOT EXISTS metadata.dqx_execution_log (
    table_name      STRING,
    city            STRING,
    execution_time  TIMESTAMP,
    total_records   BIGINT,
    passed_records  BIGINT,
    failed_records  BIGINT,
    created_date    DATE
);

In [0]:
%sql
-- Step 3: Truncate and reseed pipeline_control
TRUNCATE TABLE metadata.pipeline_control;

INSERT INTO metadata.pipeline_control VALUES
('dallas',  'Dallas.csv',  'Y', 'Dallas',  current_date(), current_date()),
('chicago', 'Chicago.csv', 'Y', 'Chicago', current_date(), current_date());

-- Step 4: Verify
SELECT * FROM metadata.pipeline_control;